In [1]:
import numpy as np
import rasterio
from pathlib import Path
from itertools import combinations
from scipy.stats import norm

CS_DIR  = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Storage_correction")
OUT_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Trend_Analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(1990, 2023))
n = len(YEARS)

# ============================================================
# 读取并堆叠所有年份，shape: (n_years, rows, cols)
# ============================================================
stack, prof = [], None
for year in YEARS:
    with rasterio.open(CS_DIR / f"CS_{year}.tif") as src:
        arr = src.read(1).astype(np.float32)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        stack.append(arr)
        if prof is None:
            prof = src.profile

stack = np.stack(stack, axis=0)   # (n_years, rows, cols)
rows, cols = stack.shape[1], stack.shape[2]

valid = ~np.any(np.isnan(stack), axis=0)
print(f"有效像元数: {valid.sum():,} / {rows*cols:,}")

X = stack[:, valid]   # (n_years, n_valid)
n_valid = X.shape[1]
del stack

# ============================================================
# 向量化 Mann-Kendall S 统计量
# ============================================================
pairs = list(combinations(range(n), 2))   # 528 对
n_pairs = len(pairs)

S = np.zeros(n_valid, dtype=np.float64)
for i, j in pairs:
    S += np.sign(X[j] - X[i])

var_S = n * (n - 1) * (2 * n + 5) / 18
Z    = np.where(S > 0, (S - 1), np.where(S < 0, (S + 1), 0)) / np.sqrt(var_S)
pval = 2 * (1 - norm.cdf(np.abs(Z)))
print("MK 计算完成")

# ============================================================
# Theil-Sen slope：分块避免单次内存峰值
# 64G 内存下 CHUNK=5_000_000，单块约 528×5M×4B ≈ 10GB
# ============================================================
CHUNK = 5_000_000
slope = np.empty(n_valid, dtype=np.float32)

for start in range(0, n_valid, CHUNK):
    end = min(start + CHUNK, n_valid)
    x_chunk = X[:, start:end]                           # (n_years, CHUNK)
    s_chunk = np.empty((n_pairs, end-start), dtype=np.float32)
    for k, (i, j) in enumerate(pairs):
        s_chunk[k] = (x_chunk[j] - x_chunk[i]) / (j - i)
    slope[start:end] = np.median(s_chunk, axis=0)
    del s_chunk
    print(f"  Theil-Sen 进度: {end:,} / {n_valid:,}")

print("Theil-Sen 计算完成")

# ============================================================
# 写回完整栅格
# ============================================================
def to_map(values):
    arr = np.full((rows, cols), np.nan, dtype=np.float32)
    arr[valid] = values.astype(np.float32)
    return arr

def save_r(arr, path):
    p = prof.copy()
    p.update(dtype="float32", count=1, nodata=-9999,
             compress="lzw", predictor=2,
             tiled=True, blockxsize=256, blockysize=256)
    out = np.where(np.isnan(arr), -9999, arr).astype(np.float32)
    with rasterio.open(path, "w", **p) as dst:
        dst.write(out, 1)

save_r(to_map(slope), OUT_DIR / "TS_slope.tif")
save_r(to_map(pval),  OUT_DIR / "MK_pvalue.tif")
save_r(to_map(Z),     OUT_DIR / "MK_Z.tif")
print("全部保存完成。")

有效像元数: 266,857,865 / 413,246,352
MK 计算完成
  Theil-Sen 进度: 5,000,000 / 266,857,865
  Theil-Sen 进度: 10,000,000 / 266,857,865
  Theil-Sen 进度: 15,000,000 / 266,857,865
  Theil-Sen 进度: 20,000,000 / 266,857,865
  Theil-Sen 进度: 25,000,000 / 266,857,865
  Theil-Sen 进度: 30,000,000 / 266,857,865
  Theil-Sen 进度: 35,000,000 / 266,857,865
  Theil-Sen 进度: 40,000,000 / 266,857,865
  Theil-Sen 进度: 45,000,000 / 266,857,865
  Theil-Sen 进度: 50,000,000 / 266,857,865
  Theil-Sen 进度: 55,000,000 / 266,857,865
  Theil-Sen 进度: 60,000,000 / 266,857,865
  Theil-Sen 进度: 65,000,000 / 266,857,865
  Theil-Sen 进度: 70,000,000 / 266,857,865
  Theil-Sen 进度: 75,000,000 / 266,857,865
  Theil-Sen 进度: 80,000,000 / 266,857,865
  Theil-Sen 进度: 85,000,000 / 266,857,865
  Theil-Sen 进度: 90,000,000 / 266,857,865
  Theil-Sen 进度: 95,000,000 / 266,857,865
  Theil-Sen 进度: 100,000,000 / 266,857,865
  Theil-Sen 进度: 105,000,000 / 266,857,865
  Theil-Sen 进度: 110,000,000 / 266,857,865
  Theil-Sen 进度: 115,000,000 / 266,857,865
  Theil-Sen 进